# 00 - Setup & smoke test

Run this notebook first, on a fresh Kaggle or Colab session. It:

1. gets `ssm_lab` importable (clone + install if needed),
2. re-proves (in this exact environment / torch version) that the parallel
   scan matches the sequential ground truth and that S4D's `forward()`
   matches its own `step()`-by-`step()` recurrence,
3. trains a tiny model on the copying task for a few seconds end-to-end.

If every cell here runs clean, every other model notebook in this repo
(`01_s4.ipynb`, and the 13 to follow) will run clean too -- they all share
this exact setup pattern.

## Setup

Run this cell first, every time. It detects Colab vs. Kaggle vs. local,
gets `ssm_lab` importable either way, and installs dependencies if needed.
Fill in `REPO_URL` below once you've pushed this repo to your own GitHub.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/YOUR_USERNAME/state-space-models.git"  # <- set this after pushing


def _in_colab() -> bool:
    return "google.colab" in sys.modules


def _in_kaggle() -> bool:
    return "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle")


IN_COLAB, IN_KAGGLE = _in_colab(), _in_kaggle()
IN_HOSTED_NOTEBOOK = IN_COLAB or IN_KAGGLE
print(f"Environment -> Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE} | Local: {not IN_HOSTED_NOTEBOOK}")


def ensure_repo_on_path() -> None:
    try:
        import ssm_lab  # noqa: F401

        print("ssm_lab already importable -- nothing to do.")
        return
    except ImportError:
        pass

    for candidate in ["state-space-models", "/kaggle/working/state-space-models", "."]:
        if os.path.isdir(os.path.join(candidate, "src", "ssm_lab")):
            sys.path.insert(0, os.path.join(candidate, "src"))
            print(f"Found repo at '{candidate}', added src/ to sys.path.")
            return

    if IN_HOSTED_NOTEBOOK:
        print("Repo not found locally -- cloning...")
        subprocess.run(["git", "clone", "--quiet", REPO_URL], check=True)
        sys.path.insert(0, "state-space-models/src")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", "state-space-models/requirements.txt"],
            check=True,
        )
    else:
        raise RuntimeError(
            "Could not find ssm_lab. Run this notebook from inside the repo "
            "(e.g. `jupyter notebook` from the repo root), or set REPO_URL "
            "above and re-run."
        )


ensure_repo_on_path()
import ssm_lab  # noqa: E402

print("ssm_lab imported OK from:", ssm_lab.__file__)


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print(
        "No GPU detected. On Kaggle: Settings -> Accelerator -> GPU T4 x2 or P100. "
        "On Colab: Runtime -> Change runtime type -> T4 GPU. "
        "CPU works fine for FAST_DEV_RUN=True."
    )


## Fast dev run

`FAST_DEV_RUN = True` shrinks every config below to a ~10-30 second CPU-friendly
sanity check (tiny model, tiny sequences, ~100 steps) -- use it to confirm the
whole notebook runs cleanly before committing GPU time to it. Set it to
`False` for the real run this notebook is meant to produce.

In [ ]:
FAST_DEV_RUN = True  # set False for the real run


## 1. Scan correctness

`parallel_scan_log` (what every model actually trains with, O(log L) sequential
rounds) must agree with `sequential_scan` (the O(L) ground truth) for both
real- and complex-valued recurrences -- S4D's state is complex, Mamba's
(landing in `02_mamba.ipynb`) is real, and this one scan implementation
serves both.

In [ ]:
import torch

from ssm_lab.utils.scan import parallel_scan_log, sequential_scan

torch.manual_seed(0)
B, L, D = 4, 65, 8  # deliberately not a power of 2

a_real = torch.rand(B, L, D) * 0.9 + 0.05
b_real = torch.randn(B, L, D)
diff_real = (sequential_scan(a_real, b_real) - parallel_scan_log(a_real, b_real)).abs().max().item()

a_complex = (torch.rand(B, L, D) * 0.9) * torch.exp(1j * torch.rand(B, L, D) * 6.28)
b_complex = torch.randn(B, L, D, dtype=torch.cfloat)
diff_complex = (sequential_scan(a_complex, b_complex) - parallel_scan_log(a_complex, b_complex)).abs().max().item()

print(f"real-valued max abs diff:    {diff_real:.2e}")
print(f"complex-valued max abs diff: {diff_complex:.2e}")
assert diff_real < 1e-4 and diff_complex < 1e-4, "scan mismatch -- stop and debug before trusting any training run"
print("Scan check: PASS")


## 2. S4D forward vs. step-by-step recurrence

Two independent code paths -- the vectorized scan-based `forward()`, and a
plain Python loop calling `step()` once per token -- computing the exact same
recurrence. Agreement here is real evidence the discretization math is
right, not just self-consistency.

In [ ]:
from ssm_lab.layers.s4d import S4DLayer

torch.manual_seed(0)
layer = S4DLayer(d_model=16, d_state=8)
x = torch.randn(2, 20, 16)

y_forward = layer(x)

state = layer.init_state(batch_size=2)
y_steps = []
for t in range(x.shape[1]):
    y_t, state = layer.step(x[:, t], state)
    y_steps.append(y_t)
y_step = torch.stack(y_steps, dim=1)

diff = (y_forward - y_step).abs().max().item()
print(f"forward() vs step-by-step max abs diff: {diff:.2e}")
assert diff < 1e-4, "S4D forward/step mismatch"
print("S4D forward == step-by-step recurrence: PASS")


## 3. End-to-end smoke test: train a tiny model on the copying task

Full pipeline: benchmark generator -> `SequenceBackbone` -> `Trainer` ->
optimizer -> eval. If loss drops and accuracy climbs here, the whole stack
this repo is built on is wired correctly.

In [ ]:
import matplotlib.pyplot as plt

from ssm_lab.benchmarks.copying import make_copy_loader
from ssm_lab.models.backbone import BackboneConfig, SequenceBackbone
from ssm_lab.training.trainer import Trainer, TrainConfig

if FAST_DEV_RUN:
    d_model, d_state, n_layers, max_steps, batch_size, seq_len, copy_len, vocab_size = 32, 16, 2, 300, 16, 10, 3, 10
else:
    d_model, d_state, n_layers, max_steps, batch_size, seq_len, copy_len, vocab_size = 128, 64, 4, 1500, 64, 64, 16, 64

cfg = BackboneConfig(vocab_size=vocab_size, d_model=d_model, n_layers=n_layers, mixer_kwargs=dict(d_state=d_state))
torch.manual_seed(0)  # reseed here so this result doesn't depend on how much randomness earlier cells consumed
model = SequenceBackbone(S4DLayer, cfg)
print(f"model params: {model.num_params():,}")

tcfg = TrainConfig(
    lr=5e-3, warmup_steps=max(10, max_steps // 15), max_steps=max_steps,
    eval_every=max(10, max_steps // 8), eval_iters=5, log_every=max(10, max_steps // 4), device=DEVICE,
)
trainer = Trainer(model, tcfg, run_name="00_smoke_test")
train_fn = make_copy_loader(batch_size, seq_len, vocab_size, copy_len, device=DEVICE, seed=1)
val_fn = make_copy_loader(batch_size, seq_len, vocab_size, copy_len, device=DEVICE, seed=2)
history = trainer.fit(train_fn, val_fn)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history["step"], history["val_loss"], marker="o")
axes[0].set(title="val loss", xlabel="step")
axes[1].plot(history["step"], history["val_acc"], marker="o", color="darkorange")
axes[1].set(title="val accuracy", xlabel="step", ylim=(0, 1.02))
fig.tight_layout()
plt.show()

print(f"final val accuracy: {history['val_acc'][-1]:.3f}")


## Done

If the plot above shows accuracy climbing well above chance
(`1/vocab_size`), the environment, the repo, and the whole training stack
are all confirmed working. Move on to `01_s4.ipynb` for the first real
benchmark sweep, or jump straight to whichever model notebook you're
picking up next -- they all start with this same setup cell.